# Model Selection and Download

In this notebook, we'll select and download models from Hugging Face for our optimization experiments. We'll focus on transformer models that are commonly used in production environments.

## 1. Import Dependencies

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM
import boto3
import time
from pathlib import Path

# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION

## 2. Define Models to Download

We'll download several models for different tasks to demonstrate optimization techniques across various model architectures and sizes.

In [ ]:
# Define models to download
models_to_download = {
    "sentiment_analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "sequence-classification",
        "description": "DistilBERT model fine-tuned for sentiment analysis"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification",
        "description": "BERT model fine-tuned for named entity recognition"
    },
    "question_answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering",
        "description": "DistilBERT model fine-tuned for question answering"
    },
    "masked_lm": {
        "model_name": "bert-base-uncased",
        "task": "masked-lm",
        "description": "BERT model for masked language modeling"
    }
}

## 3. Create Function to Download Models

In [ ]:
def download_model(model_info, save_dir="models"):
    """Download model and tokenizer from Hugging Face."""
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Create directory if it doesn't exist
    model_dir = os.path.join(save_dir, model_name.replace("/", "_"))
    os.makedirs(model_dir, exist_ok=True)
    
    print(f"Downloading {model_name} for {task}...")
    
    # Download tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.save_pretrained(model_dir)
    
    # Download model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Save model
    model.save_pretrained(model_dir)
    
    # Get model size
    model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)
    
    print(f"Downloaded {model_name} ({model_size_mb:.2f} MB)")
    
    return {
        "model_name": model_name,
        "task": task,
        "local_path": model_dir,
        "size_mb": model_size_mb
    }

## 4. Download Models

In [ ]:
# Create models directory
os.makedirs("models", exist_ok=True)

# Download models
downloaded_models = {}
for model_key, model_info in models_to_download.items():
    downloaded_models[model_key] = download_model(model_info)

## 5. Upload Models to S3

Now that we've downloaded the models, let's upload them to S3 for later use in SageMaker.

In [ ]:
def upload_model_to_s3(model_info, bucket_name):
    """Upload model files to S3."""
    s3_client = boto3.client('s3')
    local_path = model_info["local_path"]
    model_name = os.path.basename(local_path)
    s3_prefix = f"models/{model_name}"
    
    print(f"Uploading {model_name} to S3...")
    
    # Upload all files in the model directory
    for root, dirs, files in os.walk(local_path):
        for file in files:
            local_file_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_file_path, local_path)
            s3_key = f"{s3_prefix}/{relative_path}"
            
            s3_client.upload_file(local_file_path, bucket_name, s3_key)
    
    print(f"Uploaded {model_name} to s3://{bucket_name}/{s3_prefix}")
    
    return f"s3://{bucket_name}/{s3_prefix}"

In [ ]:
# Upload models to S3
for model_key, model_info in downloaded_models.items():
    s3_path = upload_model_to_s3(model_info, S3_BUCKET)
    downloaded_models[model_key]["s3_path"] = s3_path

## 6. Save Model Information

Let's save the model information for use in later notebooks.

In [ ]:
import json

# Save model information to file
with open('model_info.json', 'w') as f:
    json.dump(downloaded_models, f, indent=2)

print("Model information saved to model_info.json")

## 7. Display Model Summary

In [ ]:
import pandas as pd

# Create a DataFrame with model information
model_data = []
for model_key, model_info in downloaded_models.items():
    model_data.append({
        "Task": model_key,
        "Model": model_info["model_name"],
        "Size (MB)": f"{model_info['size_mb']:.2f}",
        "Local Path": model_info["local_path"],
        "S3 Path": model_info["s3_path"]
    })

model_df = pd.DataFrame(model_data)
model_df

## 8. Next Steps

Now that we've downloaded and uploaded our models, we're ready to proceed to the next notebook where we'll establish baseline performance metrics for each model.